# 11 — Exact Threshold Sweep, Re-Threshold Strategy, LightGBM Diagnostic

Corrects notebook 10's best-threshold MCC, whose 199-point quantile grid could miss the cut equivalent to the 0.5 operating point on extreme-imbalance corpora (underestimating the maximum by up to 0.15). The sweep here is exact: MCC is evaluated at every distinct score cut after a single sort, so it is provably no lower than MCC at any fixed threshold.

Refits the 36 source models (identical seeds and caps, so all_metrics at 0.5 reproduces notebook 10) and recomputes in_domain, zero_shot and calibrate rows with the exact sweep. Adds the rethreshold strategy: the operating point is chosen on the labelled buffer (maximising MCC of the frozen source model's scores on the buffer) and applied to the evaluation set; the buffer-side MCC at that point is recorded as buffer_est, the deployable estimate of the ceiling. Adds a LightGBM diagnostic on NF-BoT-IoT-v2 in-domain (defaults vs is_unbalance vs min_child_samples), since notebook 10 found seed-unstable in-domain failure (MCC 0.11 to 0.67) for LightGBM on that corpus only. Results append to fc_results_v2.csv with resume-skip; buffer_only and augment rows are unchanged and remain in fc_results.csv.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seeds         = [42, 43, 44],
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    budgets       = [0.0001, 0.001, 0.01, 0.05, 0.10],
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
)
MODELS = ['rf', 'lgbm', 'mlp']
V2_CSV = f'{RESULT}/fc_results_v2.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train',
        'macro_f1', 'weighted_f1', 'mcc', 'auprc_macro', 'fp_rate', 'brier', 'ece',
        'mcc_best_thr', 'thr_best', 'buffer_est', 'thr_applied', 'single_class_buffer', 'fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
for tag, d in DATASETS.items():
    assert [c for c in d.columns if c not in ('Label', 'Attack')] == FEATURES, tag
print('features:', len(FEATURES))

In [ ]:
import numpy as np

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    """Exact maximum of MCC over every distinct score cut (predict positive iff p >= t),
    including the all-negative predictor (MCC 0). Always >= MCC at any fixed threshold."""
    y = np.asarray(y_true).astype(np.int64)
    p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort')
    ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    if mccs[i] <= 0.0:
        return 0.0, float('inf')
    return float(mccs[i]), float(cuts[i])

def metrics_at_threshold(y_true, p_pos, thr):
    """MCC / macro-F1 / FP-rate at an externally chosen threshold."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true); pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred), macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(V2_CSV, mode='a', index=False, header=not os.path.exists(V2_CSV))

done = set()
if os.path.exists(V2_CSV):
    prev = pd.read_csv(V2_CSV)
    done = set(map(tuple, prev[['seed', 'source', 'target', 'model', 'budget', 'strategy']].astype(str).values))
    print(f'resume: {len(done)} rows already recorded')

def key(seed, src, tgt, m, b, s):
    return (str(seed), src, tgt, m, str(b), s)

def is_done(*k):
    return key(*k) in done

def mark(seed, src, tgt, m, b, s, metrics, n_train, **extra):
    row = dict(seed=seed, source=src, target=tgt, model=m, budget=b, strategy=s, n_train=n_train, **metrics, **extra)
    record(row)
    done.add(key(seed, src, tgt, m, b, s))
    print(f"  s{seed} {src}->{tgt} {m} b={b} {s}: MCC={metrics['mcc']:.3f} best={metrics['mcc_best_thr']:.3f}")

STRATS = ['in_domain', 'zero_shot', 'calibrate', 'rethreshold']

for seed in CFG['seeds']:
    parts = {}
    for tag, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
        parts[tag] = dict(train_full=tr.reset_index(drop=True),
                          train=stratified_cap(tr, CFG['train_cap'], seed),
                          eval=stratified_cap(te, CFG['eval_cap'], seed))
    for src in CFG['corpora']:
        others = [t for t in CFG['corpora'] if t != src]
        Xs, med_s = clean_X(parts[src]['train'], FEATURES)
        ys = parts[src]['train']['Label'].values
        for mname in MODELS:
            need = (not is_done(seed, src, src, mname, 0, 'in_domain')) or any(
                not is_done(seed, src, tgt, mname, 0, 'zero_shot') for tgt in others) or any(
                not is_done(seed, src, tgt, mname, b, s)
                for tgt in others for b in CFG['budgets'] for s in ('calibrate', 'rethreshold'))
            if not need:
                continue
            model = make_model(mname, seed, len(Xs))
            t0 = time.time()
            model.fit(Xs, ys)
            fit_s = round(time.time() - t0)
            print(f'seed {seed} | source fit {mname} on {src}: {fit_s}s')
            for tgt in CFG['corpora']:
                ev = parts[tgt]['eval']
                yev = ev['Label'].values
                Xev, _ = clean_X(ev, FEATURES, medians=med_s)
                p = model.predict_proba(Xev)[:, 1]
                del Xev
                strat = 'in_domain' if tgt == src else 'zero_shot'
                if not is_done(seed, src, tgt, mname, 0, strat):
                    mark(seed, src, tgt, mname, 0, strat, all_metrics(yev, p), len(Xs), fit_s=fit_s)
                if tgt == src:
                    continue
                for b in CFG['budgets']:
                    buf = stratified_frac(parts[tgt]['train_full'], b, seed)
                    ybuf = buf['Label'].values
                    single = int(len(np.unique(ybuf)) < 2)
                    Xb, _ = clean_X(buf, FEATURES, medians=med_s)
                    pb = model.predict_proba(Xb)[:, 1]
                    del Xb
                    if not is_done(seed, src, tgt, mname, b, 'calibrate'):
                        lr = platt_fit(pb, ybuf)
                        mark(seed, src, tgt, mname, b, 'calibrate',
                             all_metrics(yev, platt_apply(lr, p, ybuf)), len(buf), single_class_buffer=single)
                    if not is_done(seed, src, tgt, mname, b, 'rethreshold'):
                        buf_mcc, thr = best_threshold_mcc(ybuf, pb) if not single else (0.0, 0.5)
                        m = all_metrics(yev, p)
                        m.update(metrics_at_threshold(yev, p, thr))
                        mark(seed, src, tgt, mname, b, 'rethreshold', m, len(buf),
                             buffer_est=buf_mcc, thr_applied=thr, single_class_buffer=single)
            del model
            gc.collect()
        del Xs
        gc.collect()

print('rows recorded:', len(done))

In [ ]:
# LightGBM diagnostic on NF-BoT-IoT-v2 in-domain: defaults vs imbalance handling vs leaf size.
import lightgbm as lgb
DIAG_CSV = f'{RESULT}/fc_lgbm_bot_diag.csv'
variants = {
    'default':            dict(),
    'is_unbalance':       dict(is_unbalance=True),
    'min_child_5':        dict(min_child_samples=5),
    'unbalance_child_5':  dict(is_unbalance=True, min_child_samples=5),
}
rows = []
for seed in CFG['seeds']:
    d = DATASETS['nfbotv2']
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    trc = stratified_cap(tr, CFG['train_cap'], seed); evc = stratified_cap(te, CFG['eval_cap'], seed)
    Xtr, med = clean_X(trc, FEATURES); ytr = trc['Label'].values
    Xev, _ = clean_X(evc, FEATURES, medians=med); yev = evc['Label'].values
    for name, kw in variants.items():
        mdl = lgb.LGBMClassifier(n_estimators=CFG['rf_estimators'], random_state=seed, n_jobs=-1, verbosity=-1, **kw)
        t0 = time.time(); mdl.fit(Xtr, ytr)
        p = mdl.predict_proba(Xev)[:, 1]
        m = all_metrics(yev, p); m.update(seed=seed, variant=name, fit_s=round(time.time() - t0),
                                          n_benign_train=int((ytr == 0).sum()), n_trees=mdl.n_estimators_ if hasattr(mdl, 'n_estimators_') else np.nan)
        rows.append(m)
        print(f"seed {seed} {name:18s} MCC={m['mcc']:.3f} best={m['mcc_best_thr']:.3f} AUPRC={m['auprc_macro']:.3f} FPR={m['fp_rate']:.3f} {m['fit_s']}s")
    del Xtr, Xev; gc.collect()
pd.DataFrame(rows).round(4).to_csv(DIAG_CSV, index=False)
print('saved', DIAG_CSV)

In [ ]:
from scipy.stats import spearmanr, wilcoxon

v2 = pd.read_csv(V2_CSV).drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])
v1 = pd.read_csv(f'{RESULT}/fc_results.csv').drop_duplicates(['seed', 'source', 'target', 'model', 'budget', 'strategy'])

# reproduction check: metrics at 0.5 must match notebook 10 for the refit strategies
chk = v1[v1.strategy.isin(['in_domain', 'zero_shot', 'calibrate'])].merge(
    v2[v2.strategy.isin(['in_domain', 'zero_shot', 'calibrate'])],
    on=['seed', 'source', 'target', 'model', 'budget', 'strategy'], suffixes=('_v1', '_v2'))
print(f'reproduction: {len(chk)} rows, max |MCC v1 - v2| = {(chk.mcc_v1 - chk.mcc_v2).abs().max():.2e}')
print(f'sweep correction: mean(best_v2 - best_v1) = {(chk.mcc_best_thr_v2 - chk.mcc_best_thr_v1).mean():+.4f}, '
      f'rows where v2 < mcc@0.5: {int((chk.mcc_best_thr_v2 < chk.mcc_v2 - 1e-9).sum())} (must be 0)')

# combined table: v2 for refit strategies, v1 for buffer_only and augment
combined = pd.concat([v2, v1[v1.strategy.isin(['buffer_only', 'augment'])]], ignore_index=True)
combined.to_csv(f'{RESULT}/fc_results_combined.csv', index=False)

metrics = ['mcc', 'mcc_best_thr', 'macro_f1', 'auprc_macro', 'fp_rate', 'ece']
mat = combined[combined.strategy.isin(['in_domain', 'zero_shot'])]
agg = mat.groupby(['source', 'model', 'target', 'strategy'])[metrics].agg(['mean', 'std']).round(4)
agg.columns = ['_'.join(c) for c in agg.columns]
agg.reset_index().to_csv(f'{RESULT}/fc_matrix_aggregate.csv', index=False)

bud = combined[combined.strategy.isin(['zero_shot', 'calibrate', 'rethreshold', 'buffer_only', 'augment'])]
bagg = bud.groupby(['source', 'target', 'model', 'budget', 'strategy'])[metrics + ['n_train']].agg(['mean', 'std']).round(4)
bagg.columns = ['_'.join(c) for c in bagg.columns]
bagg.reset_index().to_csv(f'{RESULT}/fc_budget_aggregate.csv', index=False)

# ceiling bound: for every budget cell, rethreshold and calibrate must not exceed the exact eval ceiling
zs = v2[v2.strategy == 'zero_shot'][['seed', 'source', 'target', 'model', 'mcc_best_thr']].rename(columns={'mcc_best_thr': 'ceiling'})
rt = v2[v2.strategy == 'rethreshold'][['seed', 'source', 'target', 'model', 'budget', 'mcc', 'buffer_est']].rename(columns={'mcc': 'rt_mcc'})
ca = v2[v2.strategy == 'calibrate'][['seed', 'source', 'target', 'model', 'budget', 'mcc']].rename(columns={'mcc': 'cal_mcc'})
bo = v1[v1.strategy == 'buffer_only'][['seed', 'source', 'target', 'model', 'budget', 'mcc']].rename(columns={'mcc': 'bo_mcc'})
diag = zs.merge(rt, on=['seed', 'source', 'target', 'model']).merge(ca, on=['seed', 'source', 'target', 'model', 'budget']).merge(bo, on=['seed', 'source', 'target', 'model', 'budget'])
diag.to_csv(f'{RESULT}/fc_diagnostic.csv', index=False)
print(f'\nbound violations (rethreshold or calibrate above eval ceiling): '
      f'{int(((diag.rt_mcc > diag.ceiling + 1e-6) | (diag.cal_mcc > diag.ceiling + 1e-6)).sum())} of {len(diag)}')
print('\nper budget: mean ceiling | rethreshold realised | calibrate | buffer_only | cells where ceiling > buffer_only')
print(diag.groupby('budget').agg(ceiling=('ceiling', 'mean'), rethreshold=('rt_mcc', 'mean'), calibrate=('cal_mcc', 'mean'),
                                 buffer_only=('bo_mcc', 'mean'), ceiling_wins=('ceiling', lambda s: int((s > diag.loc[s.index, 'bo_mcc']).sum()))).round(3))
rho, pv = spearmanr(diag.buffer_est, diag.rt_mcc)
print(f'\nbuffer-side estimate vs realised rethreshold MCC: Spearman rho={rho:.3f} (p={pv:.2e}), n={len(diag)}')
for b in CFG['budgets']:
    s = diag[diag.budget == b]; r, _ = spearmanr(s.buffer_est, s.rt_mcc)
    print(f'  budget {b}: rho={r:.3f}, mean |buffer_est - realised| = {(s.buffer_est - s.rt_mcc).abs().mean():.3f}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "11: exact threshold sweep, rethreshold strategy, LightGBM BoT diagnostic; corrected aggregates"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)